In [1]:
import pandas as pd
from sqlalchemy import create_engine
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Konek ke MySQL Laragon
engine = create_engine('mysql+pymysql://root:@localhost/food_estate_db')

# Test koneksi
with engine.connect() as conn:
    print(" Koneksi database berhasil!")

# Cek semua tabel
tabel = pd.read_sql("SHOW TABLES", engine)
print("\nTabel tersedia:")
print(tabel)

 Koneksi database berhasil!

Tabel tersedia:
  Tables_in_food_estate_db
0                     desa
1               distribusi
2                komoditas
3                    panen
4                   petani


In [5]:
query = """
SELECT 
    d.id AS id_distribusi,
    d.tanggal_distribusi,
    d.tujuan,
    d.jumlah_kg AS jumlah_distribusi_kg,
    d.harga_per_kg,
    d.total_harga,
    p.id AS id_panen,
    p.tanggal_panen,
    p.kualitas,
    p.jumlah_kg AS jumlah_panen_kg,
    pet.nama AS nama_petani,
    pet.nik,
    pet.luas_lahan_ha,
    pet.tahun_bergabung,
    des.nama_desa,
    des.kecamatan,
    k.nama AS komoditas,
    k.satuan
FROM distribusi d
JOIN panen p ON d.panen_id = p.id
JOIN petani pet ON p.petani_id = pet.id
JOIN desa des ON pet.desa_id = des.id
JOIN komoditas k ON p.komoditas_id = k.id;
"""

In [6]:

df = pd.read_sql(query, engine)


df.head()

,id_distribusi,tanggal_distribusi,tujuan,jumlah_distribusi_kg,harga_per_kg,total_harga,id_panen,tanggal_panen,kualitas,jumlah_panen_kg,nama_petani,nik,luas_lahan_ha,tahun_bergabung,nama_desa,kecamatan,komoditas,satuan
0,1,2024-01-17,Bulog Palangkaraya,3000.0,5200.0,15600000.0,1,2024-01-15,A,3500.0,Ahmad Subandi,6201010101010001,2.50,2021,Kelampangan,Sebangau,Padi,kg
1,16,2024-06-12,Distributor Banjarmasin,2600.0,4800.0,12480000.0,16,2024-06-10,A,2800.0,Ahmad Subandi,6201010101010001,2.50,2021,Kelampangan,Sebangau,Jagung,kg
2,2,2024-01-22,Pasar Kahayan,1800.0,5000.0,9000000.0,2,2024-01-20,B,2100.0,Siti Rahmah,6201010101010002,1.75,2021,Kelampangan,Sebangau,Padi,kg
3,23,2024-10-03,Bulog Palangkaraya,2200.0,5200.0,11440000.0,23,2024-10-01,A,2450.0,Siti Rahmah,6201010101010002,1.75,2021,Kelampangan,Sebangau,Padi,kg
4,3,2024-02-07,Distributor Banjarmasin,4000.0,4800.0,19200000.0,3,2024-02-05,A,4200.0,Budi Santoso,6201010101010003,3.00,2022,Garantung,Maliku,Jagung,kg


In [7]:
# 1. Cleaning & Formatting Tanggal
df['tanggal_panen'] = pd.to_datetime(df['tanggal_panen'])
df['tanggal_distribusi'] = pd.to_datetime(df['tanggal_distribusi'])

# 2. Menambahkan Kolom Insight (Analisis)
# Mengambil nama bulan untuk dianalisis di Power BI nanti
df['bulan_panen'] = df['tanggal_panen'].dt.month_name()
df['bulan_distribusi'] = df['tanggal_distribusi'].dt.month_name()

# Menghitung Produktivitas: Hasil Panen / Luas Lahan
df['produktivitas_kg_per_ha'] = df['jumlah_panen_kg'] / df['luas_lahan_ha']

# Menghitung selisih (apakah ada yang tidak terjual/didistribusi?)
df['selisih_kg'] = df['jumlah_panen_kg'] - df['jumlah_distribusi_kg']

# 3. Cek hasil olahan
df.head()

,id_distribusi,tanggal_distribusi,tujuan,jumlah_distribusi_kg,harga_per_kg,total_harga,id_panen,tanggal_panen,kualitas,jumlah_panen_kg,...,luas_lahan_ha,tahun_bergabung,nama_desa,kecamatan,komoditas,satuan,bulan_panen,bulan_distribusi,produktivitas_kg_per_ha,selisih_kg
0,1,2024-01-17,Bulog Palangkaraya,3000.0,5200.0,15600000.0,1,2024-01-15,A,3500.0,...,2.50,2021,Kelampangan,Sebangau,Padi,kg,January,January,1400.0,500.0
1,16,2024-06-12,Distributor Banjarmasin,2600.0,4800.0,12480000.0,16,2024-06-10,A,2800.0,...,2.50,2021,Kelampangan,Sebangau,Jagung,kg,June,June,1120.0,200.0
2,2,2024-01-22,Pasar Kahayan,1800.0,5000.0,9000000.0,2,2024-01-20,B,2100.0,...,1.75,2021,Kelampangan,Sebangau,Padi,kg,January,January,1200.0,300.0
3,23,2024-10-03,Bulog Palangkaraya,2200.0,5200.0,11440000.0,23,2024-10-01,A,2450.0,...,1.75,2021,Kelampangan,Sebangau,Padi,kg,October,October,1400.0,250.0
4,3,2024-02-07,Distributor Banjarmasin,4000.0,4800.0,19200000.0,3,2024-02-05,A,4200.0,...,3.00,2022,Garantung,Maliku,Jagung,kg,February,February,1400.0,200.0


In [8]:
# Simpan hasil olahan ke CSV
output_path = '../output/food_estate_clean.csv'
df.to_csv(output_path, index=False)

print(f"Data berhasil disimpan di: {output_path}")

Data berhasil disimpan di: ../output/food_estate_clean.csv
